# Visualisations GPS

Ce notebook montre les fonctions de visualisation utiles pour contrôler rapidement une base GPS transformée avec `xyt_gps`.

Il reprend l'esprit de l'ancien `xyt_plot.py` tout en utilisant l'API actuelle : cartes Folium simples, couleurs déterministes, support des tables structurées et des tables exploratoires.

## 1. Préparer l'environnement

In [ ]:
import pandas as pd

import xyt_gps as xyt

## 2. Générer un petit jeu GPS de démonstration

Le jeu est synthétique et sert uniquement à vérifier le fonctionnement des visualisations. Il ne représente pas un territoire réel.

In [ ]:
phases = (
    xyt.Phase("Phase1", "2026-04-01", "2026-04-14"),
    xyt.Phase("Phase2", "2026-04-15", "2026-04-28"),
)

config = xyt.ProjectConfig(
    experiment_name="demo-visualisations",
    raw_data_dir=".",
    phases=phases,
)

synthetic = xyt.generate_synthetic_declic_gps(
    experiments=(xyt.SyntheticExperiment("demo-visualisations", phases=phases),),
    users_per_experiment=8,
    random_state=12,
    validate=True,
)

raw, dataset, indicators = xyt.run_mobility_pipeline(
    config,
    raw=synthetic.raw,
    resample_missing_days=True,
    mode_col="mode_niv1",
)

pd.DataFrame(
    [
        {"table": "legs", "rows": len(dataset.legs)},
        {"table": "staypoints", "rows": len(dataset.staypoints)},
        {"table": "storyline", "rows": len(dataset.storyline)},
    ]
)

## 3. Carte structurée depuis un `MobilityDataset`

`plot_gps_traces()` est la fonction recommandée quand les données sont déjà transformées par le package. Elle lit les `legs`, ajoute les `staypoints` si demandé et colore les traces selon une colonne métier.

In [ ]:
xyt.plot_gps_traces(
    dataset,
    sample_n=120,
    color_by="mode_niv1",
    show_staypoints=True,
    use_antpath=False,
)

## 4. Carte exploratoire sur une table quelconque

`plot_gps_on_map()` est l'adaptation de l'ancien `xyt_plot.py`. Elle est utile pour inspecter rapidement une table qui contient soit une colonne `geometry`, soit des colonnes latitude/longitude.

Elle ne remplace pas `plot_gps_traces()` pour les bases structurées, mais elle est pratique en phase de diagnostic.

In [ ]:
storyline_sample = (
    dataset.storyline
    .loc[dataset.storyline["type"].isin(["Track", "Stay"])]
    .head(160)
    .copy()
)

xyt.plot_gps_on_map(
    storyline_sample,
    type_col="type",
    color_by="user_id",
    popup_cols=("type", "user_id", "started_at", "mode_niv1", "purpose_niv1"),
)

### Variante avec colonnes latitude / longitude

Le même helper accepte aussi une table de points sans géométrie shapely, ce qui reprend l'usage simple de l'ancien `xyt_plot.py`.

In [ ]:
points = pd.DataFrame(
    [
        {"user_id": "u1", "type": "Waypoint", "latitude": 46.52, "longitude": 6.60},
        {"user_id": "u1", "type": "Waypoint", "latitude": 46.53, "longitude": 6.62},
        {"user_id": "u2", "type": "Staypoint", "latitude": 46.51, "longitude": 6.58},
    ]
)

xyt.plot_gps_on_map(points, geo_columns=("latitude", "longitude"), color_by="user_id")

## 5. Graphes de motifs journaliers

Les graphes de motifs résument la structure quotidienne des lieux visités. Le nœud `0` est le premier lieu de la séquence canonique ; les flèches indiquent les transitions entre lieux.

In [ ]:
motifs = xyt.build_mobility_motifs(dataset)
summary = xyt.summarize_mobility_motifs(motifs)
summary.head()

In [ ]:
xyt.plot_mobility_motif_graphs(
    motifs,
    top_n=6,
    title="Motifs journaliers fréquents",
)

## À retenir

- `plot_gps_traces()` : contrôle cartographique principal pour un `MobilityDataset`.
- `plot_gps_on_map()` : contrôle exploratoire pour une table GPS plus brute ou hétérogène.
- `plot_mobility_motif_graphs()` : lecture structurelle des journées sous forme de graphes de lieux canonisés.